# ☀️ Solar API 대화 요약 (v7 - Embedding Retrieval)

## 주요 변경사항 (v5 → v7)
| 항목 | v5 (이전) | v7 (현재) |
|------|----------|----------|
| Few-shot 선택 | TF-IDF Topic 기반 | **Embedding Retrieval** (multilingual-e5-base) |
| API 호출 수 | 3회/샘플 (topic+역할+요약) | **1회/샘플** (요약만) |
| 역할 추론 | infer_roles() 사용 | **제거** (EDA: 82% 태그 유지) |
| Topic 예측 | predict_topic() 사용 | **제거** |
| 생성 길이 제한 | 없음 | **max_tokens=60, 20단어 후처리** |
| System Prompt | 태그 유지 강조 | **압축 중심 + 행동 형식 지정** |


## ⚙️ 1. 환경 설정

In [ ]:
# 처음 한 번만 실행
!pip install openai sentence-transformers scikit-learn rouge python-mecab-ko -q
!apt-get install -y mecab mecab-ipadic-utf8 libmecab-dev -q


In [ ]:
import pandas as pd
import os
import re
import time
import numpy as np
from tqdm import tqdm
from rouge import Rouge
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print('라이브러리 로드 완료')


## ⚙️ 2. API & 경로 설정

In [ ]:
UPSTAGE_API_KEY = "up_opnjqOc0T694rx1It5hDPhfZNCiW5"

client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url="https://api.upstage.ai/v1/solar"
)


In [ ]:
DATA_PATH   = "/data/ephemeral/home/data/"
RESULT_PATH = "/data/ephemeral/home/code/prediction/"
os.makedirs(RESULT_PATH, exist_ok=True)


## ⚙️ 3. 데이터 로드

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_PATH, 'train.csv'))
val_df   = pd.read_csv(os.path.join(DATA_PATH, 'dev.csv'))
test_df  = pd.read_csv(os.path.join(DATA_PATH, 'test.csv'))

print(f"train: {len(train_df)}개")
print(f"val  : {len(val_df)}개")
print(f"test : {len(test_df)}개")
print(f'컬럼: {train_df.columns.tolist()}')


In [ ]:
train_df.tail()


In [ ]:
val_df.tail()


In [ ]:
test_df.tail()


## 1. 평가지표 & Embedding 인덱스

In [ ]:
rouge = Rouge()

def compute_metrics(pred, gold):
    """ROUGE-1/2/L F1 평균 반환 (공백 기반 빠른 확인용)"""
    try:
        results = rouge.get_scores(str(pred), str(gold), avg=True)
        r1 = results['rouge-1']['f']
        r2 = results['rouge-2']['f']
        rl = results['rouge-l']['f']
        return {'rouge-1': r1, 'rouge-2': r2, 'rouge-l': rl, 'avg': (r1+r2+rl)/3}
    except:
        return {'rouge-1': 0, 'rouge-2': 0, 'rouge-l': 0, 'avg': 0}


## 2. Embedding 인덱스 구축 (최초 1회, 약 5~10분)

### v5 TF-IDF vs v7 Embedding 비교
| 방식 | 문제점 | 개선 |
|------|--------|------|
| TF-IDF Topic | "저녁 파티 초대" → "저녁 식사 초대" 예측 오류 → 엉뚱한 few-shot | - |
| **Embedding** | - | 대화 내용 자체로 의미 유사도 계산 → 정확한 few-shot |

> `intfloat/multilingual-e5-base`: 한국어 성능 우수, CPU 실행 (VRAM 추가 사용 없음)  
> e5 모델은 `"query: "` / `"passage: "` prefix 필수


In [ ]:
# ✅ 한 번만 실행 - 약 5~10분 소요
embed_model = SentenceTransformer(
    "intfloat/multilingual-e5-base",
    device="cpu"
)

print("Train dialogue embedding 인코딩 시작...")
train_embeddings = embed_model.encode(
    ["passage: " + str(d) for d in train_df["dialogue"]],
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f"✅ Embedding 인덱스 구축 완료: {train_embeddings.shape}")


## 3. Embedding Retrieval 함수

In [ ]:
def get_few_shots_by_embedding(dialogue, n=3):
    """
    Embedding 유사도로 train에서 few-shot 선택
    - top5 뽑고 그 중 3개 랜덤 선택 → 다양성 증가
    - v5 TF-IDF topic 기반 대비 retrieval 정확도 향상
    """
    query_vec = embed_model.encode(
        ["query: " + str(dialogue)],
        convert_to_numpy=True
    )
    sims     = cosine_similarity(query_vec, train_embeddings).flatten()
    top5_idx = sims.argsort()[::-1][:5]
    selected = np.random.choice(top5_idx, min(n, len(top5_idx)), replace=False)
    return train_df.iloc[selected].to_dict("records")


# ✅ 테스트
sample_shots = get_few_shots_by_embedding(train_df.iloc[0]["dialogue"])
print(f"retrieval 테스트: {len(sample_shots)}개 반환")
for i, shot in enumerate(sample_shots):
    print(f"  [{i+1}] topic={shot.get('topic','없음')} | 요약: {shot['summary'][:50]}")


## 4. 핵심 함수 정의

### build_prompt & solar_summarize
- **역할 추론 제거**: EDA 결과 82%가 #Person 태그 유지 → 역할 치환 시 오히려 손해
- **압축 중심 프롬프트**: 재서술이 아닌 핵심 행동 1개 추출
- **행동 형식 지정**: 정답 패턴(A가 B에게 ~한다) 강제
- **max_tokens=60 + 20단어 후처리**: precision 보호


In [ ]:
# ============================================================
# System Prompt - 압축 중심 (v7)
# ============================================================
SYSTEM_PROMPT = (
    "당신은 한국어 대화 요약 전문가입니다.\n\n"
    "주어진 대화의 핵심 사건을 한국어 문어체로 1문장으로 요약하세요.\n\n"
    "규칙:\n"
    "1. 반드시 1문장으로만 작성합니다.\n"
    "2. 20단어 이하로 작성합니다.\n"
    "3. 대화의 가장 핵심 사건 1개만 포함합니다.\n"
    "4. 세부 설명, 이유, 결과, 추가 정보 등 불필요한 내용은 포함하지 않습니다.\n"
    "5. #Person1#, #Person2# 등 화자 태그는 절대 변경하지 않습니다.\n"
    "6. 이름이 대화에 명시적으로 등장한 경우에만 이름을 사용합니다.\n\n"
    "요약 형태 (다음 중 하나를 따르세요):\n"
    "- #Person1#이 #Person2#에게 ~을 제안한다.\n"
    "- #Person1#은 #Person2#에게 ~을 요청한다.\n"
    "- #Person1#과 #Person2#는 ~에 대해 이야기한다.\n"
    "- #Person1#이 #Person2#에게 ~을 묻는다.\n"
    "- #Person1#이 #Person2#에게 ~을 설명한다.\n\n"
    "대화 전체를 설명하지 말고 핵심 행동만 요약하세요."
)


def build_prompt(dialogue):
    """
    Embedding retrieval few-shot 포함 프롬프트 구성 (v7)
    - predict_topic(), infer_roles() 제거 → API 호출 3→1
    - role_hint 완전 제거 (EDA 결과 기반)
    - few-shot: user/assistant 교차 형식
    """
    few_shots = get_few_shots_by_embedding(dialogue, n=3)

    # system
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    # few-shot examples (user/assistant 교차)
    for ex in few_shots:
        messages.append({
            "role": "user",
            "content": f"대화:\n{ex['dialogue']}"
        })
        messages.append({
            "role": "assistant",
            "content": ex["summary"]
        })

    # target
    messages.append({
        "role": "user",
        "content": (
            f"대화:\n{dialogue}\n\n"
            "위 대화를 #Person 태그를 그대로 유지하여 반드시 1문장, 20단어 이하로 요약하세요:\n요약:"
        )
    })
    return messages


def summarization(dialogue, max_retries=3):
    """
    Solar API 요약 생성 (v7)
    - 압축 중심: temperature=0.2, max_tokens=60
    - 20단어 초과 후처리
    - 재시도 로직 유지
    """
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="solar-1-mini-chat",
                messages=build_prompt(dialogue),
                temperature=0.2,   # 낮게 → 압축적 생성
                top_p=0.9,
                max_tokens=60      # ✅ 생성 길이 강제 제한
            )
            result = response.choices[0].message.content.strip()

            # ✅ 후처리: 20단어 초과 시 자르기 (ROUGE precision 보호)
            words = result.split()
            if len(words) > 20:
                result = " ".join(words[:20])

            return result

        except Exception as e:
            print(f"  ⚠️ summarization 시도 {attempt+1}/{max_retries} 실패: {type(e).__name__}")
            if attempt < max_retries - 1:
                wait = (attempt + 1) * 10
                print(f"  {wait}초 후 재시도...")
                time.sleep(wait)
            else:
                print(f"  ❌ 최종 실패 - 빈 문자열 반환")
                return ""


## 3. 샘플 테스트

In [ ]:
def test_on_train_data(num_samples=3):
    """train 데이터로 빠른 품질 확인"""
    samples = train_df.sample(num_samples, random_state=42)
    for _, row in samples.iterrows():
        dialogue = row['dialogue']
        gold     = row['summary']
        pred     = summarization(dialogue)
        score    = compute_metrics(pred, gold)

        print(f"Dialogue (앞 200자):")
        print(dialogue[:200])
        print(f"\n예측: {pred}")
        print(f"정답: {gold}")
        print(f"[ROUGE] R1={score['rouge-1']:.3f} R2={score['rouge-2']:.3f} RL={score['rouge-l']:.3f} AVG={score['avg']:.3f}")
        print("="*80)


In [ ]:
if __name__ == "__main__":
    test_on_train_data()


## 4. Validation 성능 평가

- `validate(10)` → 빠른 확인 (약 2~3분)
- `validate(50)` → 중간 확인
- `validate(0)`  → 전체 499개


In [ ]:
def validate(num_samples=50):
    """
    dev.csv로 ROUGE 평가 + 실패 케이스 저장 (v7 - MeCab 기반)
    - API 호출: 1회/샘플 (v5 대비 3배 빠름)
    """
    from mecab import MeCab
    m = MeCab()

    def mecab_tokenize(text):
        tokens = [tok for tok, _ in m.pos(str(text)) if tok.strip()]
        return ' '.join(tokens) if tokens else str(text)

    val_samples = val_df[:num_samples] if num_samples > 0 else val_df

    results_list = []
    start_time   = time.time()

    for idx, row in tqdm(val_samples.iterrows(), total=len(val_samples)):
        dialogue = row['dialogue']
        gold     = row['summary']
        pred     = summarization(dialogue)

        # MeCab 기반 ROUGE (대회 공식 기준)
        pred_tok = mecab_tokenize(pred)
        gold_tok = mecab_tokenize(gold)

        try:
            mecab_scores = rouge.get_scores(pred_tok, gold_tok)[0]
            r1 = mecab_scores['rouge-1']['f']
            r2 = mecab_scores['rouge-2']['f']
            rl = mecab_scores['rouge-l']['f']
            avg = (r1 + r2 + rl) / 3
        except:
            r1 = r2 = rl = avg = 0.0

        results_list.append({
            'fname'   : row['fname'],
            'dialogue': dialogue,
            'gold'    : gold,
            'pred'    : pred,
            'rouge1'  : r1,
            'rouge2'  : r2,
            'rougeL'  : rl,
            'avg'     : avg,
        })

        # RPM 제한 방지 (v7: API 1회/샘플 → 60개마다 대기)
        if (idx + 1) % 60 == 0:
            elapsed = time.time() - start_time
            if elapsed < 60:
                wait = 60 - elapsed + 5
                print(f"RPM 제한 방지 대기: {wait:.0f}초")
                time.sleep(wait)
            start_time = time.time()

    result_df = pd.DataFrame(results_list)

    # 점수 출력
    print('\n' + '='*60)
    print('Validation ROUGE 점수 (MeCab 형태소 기반)')
    print('='*60)
    print(f'  ROUGE-1 : {result_df["rouge1"].mean():.4f}')
    print(f'  ROUGE-2 : {result_df["rouge2"].mean():.4f}')
    print(f'  ROUGE-L : {result_df["rougeL"].mean():.4f}')
    print(f'  평균    : {result_df["avg"].mean():.4f}')
    print('='*60)
    print(f'\n전체 평균 ROUGE: {result_df["avg"].mean():.4f}')
    print(f'하위 25%  ROUGE: {result_df["avg"].quantile(0.25):.4f}')
    print(f'상위 25%  ROUGE: {result_df["avg"].quantile(0.75):.4f}')

    # 저장
    os.makedirs(RESULT_PATH, exist_ok=True)
    save_path = os.path.join(RESULT_PATH, 'solar_val_results_v7.csv')
    result_df.to_csv(save_path, index=False)
    print(f'\n결과 저장: {save_path}')

    return result_df


In [ ]:
if __name__ == "__main__":
    val_result = validate(10)  # 빠른 확인 먼저


In [ ]:
def analyze_failures(result_df, top_n=10):
    """ROUGE 낮은 케이스 분석"""
    sorted_df = result_df.sort_values('avg')

    print(f'전체 평균 ROUGE: {result_df["avg"].mean():.4f}')
    print(f'하위 25%  ROUGE: {result_df["avg"].quantile(0.25):.4f}')
    print(f'상위 25%  ROUGE: {result_df["avg"].quantile(0.75):.4f}')

    print(f'\n{"-"*70}')
    print(f'하위 {top_n}개 실패 케이스 분석')
    print('-'*70)

    for _, row in sorted_df.head(top_n).iterrows():
        print(f'\n[{row["fname"]}] AVG={row["avg"]:.3f}')
        print(f'정답: {row["gold"]}')
        print(f'예측: {row["pred"]}')
        print('-'*70)

    # 패턴 분석
    print('\n[패턴 분석]')
    pred_lens = result_df['pred'].apply(lambda x: len(str(x).split()))
    gold_lens = result_df['gold'].apply(lambda x: len(str(x).split()))
    print(f'  예측 평균 길이: {pred_lens.mean():.1f}단어  (목표: 15단어 이하)')
    print(f'  정답 평균 길이: {gold_lens.mean():.1f}단어')

    # 태그 유지율
    pred_has_tag = result_df['pred'].apply(
        lambda x: bool(re.search(r'#Person\d+#', str(x)))
    )
    gold_has_tag = result_df['gold'].apply(
        lambda x: bool(re.search(r'#Person\d+#', str(x)))
    )
    print(f'  예측 태그 유지율: {pred_has_tag.mean()*100:.1f}%')
    print(f'  정답 태그 유지율: {gold_has_tag.mean()*100:.1f}%')


In [ ]:
if __name__ == '__main__':
    analyze_failures(val_result, top_n=10)


## 5. Test 추론 및 제출 파일 생성

> test.csv 499개 전체 추론 (v7: API 1회/샘플 → 약 10~15분)


In [ ]:
def inference():
    """test.csv 전체 추론 -> output_solar_v7.csv 저장"""
    summaries  = []
    start_time = time.time()

    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        dialogue = row['dialogue']
        summary  = summarization(dialogue)
        summaries.append(summary)

        # RPM 제한 방지 (v7: 1회/샘플 → 60개마다 대기)
        if (idx + 1) % 60 == 0:
            elapsed = time.time() - start_time
            if elapsed < 60:
                wait = 60 - elapsed + 5
                print(f"RPM 제한 방지 대기: {wait:.0f}초")
                time.sleep(wait)
            start_time = time.time()

    output = pd.DataFrame({
        "fname"  : test_df['fname'],
        "summary": summaries,
    })

    os.makedirs(RESULT_PATH, exist_ok=True)
    save_path = os.path.join(RESULT_PATH, "output_solar_v7.csv")
    output.to_csv(save_path, index=False)
    print(f"\n✅ 저장 완료: {save_path}")
    print(f'총 {len(output)}개 | 고유 요약: {output["summary"].nunique()}개')
    print(output.head(10).to_string())
    return output


In [ ]:
if __name__ == "__main__":
    output = inference()


## 5-1. 중간에 끊긴 경우 이어서 추론

In [ ]:
def inference_resume():
    """중간에 끊겼을 때 이어서 추론"""
    save_path = os.path.join(RESULT_PATH, 'output_solar_v7.csv')

    if os.path.exists(save_path):
        done_df     = pd.read_csv(save_path)
        done_fnames = set(done_df['fname'].tolist())
        print(f'이미 완료: {len(done_fnames)}개 / 전체: {len(test_df)}개')
    else:
        done_df     = pd.DataFrame(columns=['fname', 'summary'])
        done_fnames = set()

    summaries  = done_df.to_dict('records')
    start_time = time.time()

    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        if row['fname'] in done_fnames:
            continue

        dialogue = row['dialogue']
        summary  = summarization(dialogue)
        summaries.append({'fname': row['fname'], 'summary': summary})

        if (idx + 1) % 60 == 0:
            elapsed = time.time() - start_time
            if elapsed < 60:
                time.sleep(60 - elapsed + 5)
            start_time = time.time()

            # 중간 저장
            pd.DataFrame(summaries).to_csv(save_path, index=False)
            print(f'중간 저장: {len(summaries)}개')

    output = pd.DataFrame(summaries)
    output.to_csv(save_path, index=False)
    print(f'✅ 완료! {len(output)}개 | 고유 요약: {output["summary"].nunique()}개')
    return output


if __name__ == '__main__':
    output = inference_resume()


## 6. 결과 확인

In [ ]:
# test 추론 결과 확인
output = pd.read_csv(os.path.join(RESULT_PATH, 'output_solar_v7.csv'))
print(f"총 행 수 : {len(output)}")
print(f"고유 요약 수 : {output['summary'].nunique()}")
print(f"평균 길이: {output['summary'].str.split().str.len().mean():.1f}단어")
print(f"#Person 포함: {output['summary'].str.contains('#Person').mean():.1%}")
print(output.head(10).to_string())


In [ ]:
# val 평가 결과 확인
try:
    val_result = pd.read_csv(os.path.join(RESULT_PATH, 'solar_val_results_v7.csv'))
    print(f'Validation 샘플 수: {len(val_result)}')
    print(f'ROUGE-1 평균: {val_result["rouge1"].mean():.4f}')
    print(f'ROUGE-2 평균: {val_result["rouge2"].mean():.4f}')
    print(f'ROUGE-L 평균: {val_result["rougeL"].mean():.4f}')
    print(f'AVG     평균: {val_result["avg"].mean():.4f}')
    analyze_failures(val_result, top_n=5)
except FileNotFoundError:
    print('validate() 먼저 실행하세요!')
